# Project Forge — ComfyUI Remote Server

Este notebook transforma o Google Colab em um servidor ComfyUI gratuito com GPU.

**GPU:** T4 (12GB VRAM) — fornecida gratuitamente pelo Google.

---

## Como usar:
1. Menu `Runtime` → `Change runtime type` → `T4 GPU`
2. Execute cada célula em ordem
3. Quando a URL do túnel aparecer, copie (Ctrl+C) — o Forge detecta automaticamente

---

In [ ]:
# @title 1. Verificar GPU
import torch, psutil, platform, subprocess, os, json, time, threading, urllib.request

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu.name}")
    print(f"VRAM: {round(gpu.total_mem / 1024**3, 1)} GB")
else:
    print("❌ GPU não detectada. Vá em Runtime > Change runtime type > T4 GPU")
    raise SystemExit()

In [ ]:
# @title 2. Instalar dependências do sistema
!apt-get update -qq -y
!apt-get install -qq -y git wget unzip zip libgl1-mesa-glx libglib2.0-0 libsm6 libxext6 libxrender-dev libgomp1 > /dev/null 2>&1

# Instalar cloudflared (túnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print("✅ Dependências instaladas")

In [ ]:
# @title 3. Baixar/Instalar ComfyUI
COMFY_DIR = "/content/ComfyUI"

if not os.path.exists(COMFY_DIR):
    print("Baixando ComfyUI...")
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git "{COMFY_DIR}" 2>&1 | tail -1
else:
    print("ComfyUI já existe")

# Instalar requirements
print("Instalando dependências Python...")
!pip install -q -r "{COMFY_DIR}/requirements.txt" 2>&1 | tail -1

print("✅ ComfyUI pronto")

In [ ]:
# @title 4. Baixar modelo SD1.5
MODEL_NAME = "dreamshaper_8.safetensors"
MODEL_PATH = f"{COMFY_DIR}/models/checkpoints/{MODEL_NAME}"

if not os.path.exists(MODEL_PATH):
    print("Baixando Dreamshaper 8 (~2GB)...")
    urls = [
        "https://civitai.com/api/download/models/128713?type=Model&format=SafeTensor&size=pruned&fp=fp16",
        "https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors"
    ]
    ok = False
    for url in urls:
        try:
            !wget -q --show-progress -O "{MODEL_PATH}" "{url}"
            if os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) > 1_000_000_000:
                ok = True
                break
            else:
                print("Falhou, tentando outra fonte...")
        except Exception as e:
            print(f"Erro na fonte {url}: {e}")
    if not ok:
        print("❌ Nao foi possivel baixar o modelo.")
        raise SystemExit()
    print("✅ Modelo baixado")
else:
    print("✅ Modelo já existe")

print(f"\nModelo: {MODEL_NAME}")
print(f"Tamanho: {round(os.path.getsize(MODEL_PATH) / 1024**3, 1)} GB")

In [ ]:
# @title 5. (Opcional) Baixar VAE e LoRA extras
# Descomente se quiser modelos extras

# VAE melhorado
# vae_url = "https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors"
# !wget -q --show-progress -O "{COMFY_DIR}/models/vae/vae-ft-mse.safetensors" "{vae_url}"

# ControlNet (opcional)
# !wget -q --show-progress ...

In [ ]:
# @title 6. Iniciar servidor ComfyUI
import subprocess, sys

server_port = 8188

log_file = open('/content/comfyui.log', 'w')

server = subprocess.Popen(
    [sys.executable, "main.py", f"--port={server_port}", "--listen=0.0.0.0"],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True
)

print(f"✅ ComfyUI iniciado (PID: {server.pid})")
print("Aguardando servidor ficar pronto...")

# Aguardar servidor ficar acessível
import time as tmod
for i in range(120):
    try:
        req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
        urllib.request.urlopen(req, timeout=2)
        print(f"✅ Servidor pronto após {i+1}s")
        break
    except:
        tmod.sleep(1)
else:
    print("❌ Servidor não iniciou. Verifique os logs em /content/comfyui.log")
    server.terminate()
    raise SystemExit()

In [ ]:
# @title 7. Criar túnel público com Cloudflare
import urllib.request, json, subprocess, threading, time

# Iniciar cloudflared em background
tunnel_log = open('/content/tunnel.log', 'w')
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Extrair URL do log
tunnel_url = None
def read_tunnel_output():
    global tunnel_url
    for line in tunnel.stdout:
        print(f"[tunnel] {line.strip()}")
        if "https://" in line and ".trycloudflare.com" in line:
            for word in line.split():
                if "https://" in word and ".trycloudflare.com" in word:
                    tunnel_url = word.rstrip(".")

threading.Thread(target=read_tunnel_output, daemon=True).start()

# Aguardar URL
for _ in range(60):
    if tunnel_url:
        break
    time.sleep(1)

if tunnel_url:
    print(f"\n{'='*60}")
    print(f"🔗 URL DO TÚNEL: {tunnel_url}")
    print(f"{'='*60}")
else:
    print("⚠️  Cloudflare pode estar bloqueado. Tentando alternativas...")

In [ ]:
# @title 8. (Fallback) Alternativa com LocalTunnel se Cloudflare falhar
if not tunnel_url:
    print("Usando LocalTunnel como fallback...")
    !npm install -g -q localtunnel 2>/dev/null
    !npx localtunnel --port 8188 &
    time.sleep(5)
    print("\nAbra o link do LocalTunnel acima e confirme.")
    
    # Tentar extrair URL
    import requests as reqs
    try:
        r = reqs.get("https://ltportal.com/api/assign")
        tunnel_url = r.json().get("url", "") + ":8188" if r.ok else ""
    except:
        print("Abra manualmente o link mostrado acima")
else:
    print("✅ Cloudflare funcionou, sem necessidade de fallback.")

In [ ]:
# @title 9. Exibir instruções finais
if tunnel_url:
    print(f"""
╔══════════════════════════════════════════════════════════╗
║           PROJECT FORGE — COMFYUI REMOTE              ║
╠══════════════════════════════════════════════════════════╣
║                                                          ║
║  🖥️  Servidor rodando com GPU T4 gratuita               ║
║                                                          ║
║  🔗 URL: {tunnel_url:<48}║
║                                                          ║
║  📝 Copie a URL (Ctrl+C):                        ║
║     o Forge detecta automaticamente e conecta      ║
║                                                          ║
║  ⚠️  Este notebook precisa ficar rodando.              ║
║     Feche o navegador mas NÃO pare a sessão.           ║
║  ⚠️  A sessão expira em ~12h na GPU gratuita.          ║
║     Quando expirar, execute novamente.                 ║
║                                                          ║
╚══════════════════════════════════════════════════════════╝""")
else:
    print("""
╔══════════════════════════════════════════════════════════╗
║           TÚNEL NÃO DETECTADO AUTOMATICAMENTE         ║
╠══════════════════════════════════════════════════════════╣
║  1. Veja a célula acima para o link do LocalTunnel.   ║
║  2. Abra o link, confirme o acesso.                   ║
║  3. A URL final será: {URL_mostrada}:8188              ║
║  4. Copie a URL (Ctrl+C) — o Forge detecta     ║
╚══════════════════════════════════════════════════════════╝""")

In [ ]:
# @title ⏳ Keep Alive (evita desconexão)
# Esta célula mantém o notebook ativo.
# Execute se for usar por mais de 30min.

import time, threading, requests

def ping_loop():
    while True:
        try:
            requests.get("https://www.google.com", timeout=10)
        except:
            pass
        time.sleep(60)

if tunnel_url:
    t = threading.Thread(target=ping_loop, daemon=True)
    t.start()
    print("✅ Keep alive ativo (ping a cada 60s)")
else:
    print("Configure o túnel primeiro")